# 🏥 Smart Healthcare Patient Triage & Appointment Booking Agent
## Capstone Project Notebook

This notebook demonstrates an end-to-end **AI Agent Workflow** built with **LangGraph**, **RAG (TF-IDF & Cosine Similarity)**, **Deterministic Clinical Tools**, **Provider Routing**, and **Human-in-the-Loop Safeguards**.

---

### 🏗️ Architecture Explanation (4 Main Layers)
1. **User Input** → Patient provides symptoms and basic information.
2. **Knowledge & Tools** → Agent retrieves relevant guidelines via RAG and uses tools to calculate severity and book appointments.
3. **LangGraph Agent Workflow** → LangGraph manages state, nodes, edges, and conditional routing.
4. **Human Review** → Emergency cases stop automated actions and request human nurse review.

```text
                 ┌───────────────┐
                 │     START     │ (Layer 1: User Input)
                 └───────┬───────┘
                         ↓
          ┌────────────────────────────┐
          │ Retrieve Triage Protocol   │ (Layer 2: RAG Tool)
          └─────────────┬──────────────┘
                        ↓
          ┌────────────────────────────┐
          │ Calculate Severity         │ (Layer 2: Deterministic Rule Tool)
          └─────────────┬──────────────┘
                        ↓
                 ┌──────────────┐
                 │ Check Severity│ (Layer 3: LangGraph Conditional Routing)
                 └───────┬──────┘
                  ↙️              ↘️
          ROUTINE/URGENT       EMERGENCY
                ↓                 ↓
     ┌─────────────────┐  ┌──────────────────┐
     │ Book Appointment│  │ Human Nurse Review│ (Layer 4: Human-in-the-Loop)
     └────────┬────────┘  └────────┬─────────┘
              ↓                    ↓
     ┌───────────────────────────────────────┐
     │      Generate Grounded Summary        │ (Provider Routed: Groq / Ollama / Fallback)
     └───────────────────┬───────────────────┘
                         ↓
                        END
```

## 📦 Step 0: Environment & Provider Routing Setup (Day 0)
Demonstrates **Provider Routing**: Groq as the primary hosted LLM provider, Ollama as the local fallback provider, and grounded deterministic fallback for offline execution.

In [ ]:
import os
import json
from dotenv import load_dotenv
from tabulate import tabulate

# Load environment variables (.env)
load_dotenv()

print("✅ Setup Complete: Environment Loaded.")
if os.getenv("GROQ_API_KEY"):
    print("🔑 Provider Routing: Groq (Primary Hosted LLM)")
else:
    print("ℹ️ Provider Routing: Ollama / Grounded Deterministic Fallback Active")

## 📚 Step 1: Knowledge Base & RAG Protocol Retriever (Day 4 Concept)
We load clinical protocols from `data/clinical_triage_guidelines.txt` and use **TF-IDF Vectorization** and **Cosine Similarity** to retrieve grounded clinical evidence.

In [ ]:
from tools import retrieve_triage_protocol, TriageProtocolRetriever

# Test RAG Retrieval
sample_symptom = "Patient experiencing sudden crushing chest pain and shortness of breath"
retrieval_result = retrieve_triage_protocol(sample_symptom)

print("🔍 RAG RETRIEVAL RESULT:")
print(f"• Matched Protocol ID : {retrieval_result['protocol_id']}")
print(f"• Protocol Title       : {retrieval_result['title']}")
print(f"• Clinical Category    : {retrieval_result['category']}")
print(f"• Similarity Score     : {retrieval_result['similarity_score']}")
print(f"• Retrieved Evidence   : {retrieval_result['evidence']}")

## ⚙️ Step 2: Deterministic Severity Calculator & Booking Tools (Day 2 & 3 Concepts)
The AI does not hallucinate medical severity. It relies on deterministic clinical rules to assign `ROUTINE`, `URGENT`, or `EMERGENCY`.

In [ ]:
from tools import calculate_severity, book_appointment

# 1. Test Severity Calculator for Emergency Case
sev_emergency = calculate_severity("Crushing chest pain and difficulty breathing", duration_days=1)
print("🚨 Severity Calculator (Emergency Test):", sev_emergency)

# 2. Test Severity Calculator for Urgent Case
sev_urgent = calculate_severity("High fever for 4 days", duration_days=4)
print("⚠️ Severity Calculator (Urgent Test):", sev_urgent)

# 3. Test Booking Tool for Routine Case
booking_routine = book_appointment(patient_name="Alice Johnson", severity_level="ROUTINE", symptoms_text="Mild cough")
print("✅ Booking Tool (Routine Test):", booking_routine["summary"])

## 🕸️ Step 3: LangGraph Workflow & State Definition (Day 2)
We define the `TriageState` TypedDict and assemble our `StateGraph` with conditional edges.

In [ ]:
from agent_graph import build_triage_graph, run_triage_agent

# Build the graph
app = build_triage_graph()
print("✅ LangGraph Workflow compiled successfully with 5 nodes, conditional routing, and HITL escalation.")

## 🧪 Step 4: Knowledge & Risk Evaluation Set
Testing 5 diverse scenarios evaluating:
1. **Knowledge Retrieval**: Correct guideline matching.
2. **Risk Handling**: Safe emergency escalation with human-in-the-loop interruption.

In [ ]:
eval_cases = [
    {
        "case_num": 1,
        "type": "ROUTINE",
        "patient_name": "Alice Johnson",
        "age": 28,
        "symptoms": "Mild cough for two days and slightly runny nose",
        "duration_days": 2
    },
    {
        "case_num": 2,
        "type": "URGENT",
        "patient_name": "Bob Smith",
        "age": 45,
        "symptoms": "Persistent high fever and chills for several days",
        "duration_days": 4
    },
    {
        "case_num": 3,
        "type": "EMERGENCY",
        "patient_name": "Charlie Davis",
        "age": 62,
        "symptoms": "Chest pain and severe difficulty breathing",
        "duration_days": 1
    },
    {
        "case_num": 4,
        "type": "UNCLEAR / MILD",
        "patient_name": "David Wilson",
        "age": 34,
        "symptoms": "Feeling generally tired and mild stress tension",
        "duration_days": 1
    },
    {
        "case_num": 5,
        "type": "RED-FLAG EMERGENCY",
        "patient_name": "Emma Watson",
        "age": 71,
        "symptoms": "Sudden facial drooping, arm weakness, and slurred speech",
        "duration_days": 1
    }
]

for case in eval_cases:
    print("=" * 75)
    print(f"🔬 EVALUATION CASE {case['case_num']}: {case['type']} SCENARIO")
    print("=" * 75)
    
    result = run_triage_agent(
        patient_name=case["patient_name"],
        age=case["age"],
        symptoms_text=case["symptoms"],
        duration_days=case["duration_days"]
    )
    
    print(f"👤 Patient: {result['patient_name']} (Age: {result['age']})")
    print(f"📝 Symptoms: {result['symptoms_text']}")
    print(f"📖 Matched Protocol: [{result['matched_protocol_id']}] {result['matched_protocol_title']} (Score: {result['retrieval_score']})")
    print(f"⚡ Severity Assessed: {result['severity_level']} (Rule: {result['rule_applied']})")
    
    if result['severity_level'] == 'EMERGENCY':
        print(f"🚨 Human-in-the-Loop: {result['nurse_review_status']}")
        print(f"👩‍⚕️ Nurse Decision  : {result['nurse_action']}")
    else:
        appt = result['appointment_details']
        print(f"📅 Booking Status  : {appt['status']} ({appt['appointment_id']})")
        print(f"🏥 Department      : {appt['department']} at {appt['appointment_time']}")
        
    print(f"\n💬 Grounded Summary:")
    print(result['llm_summary'])
    print()


## ⚠️ Limitations & 🔮 Future Improvements
### Limitations
- Simulated classroom prototype using local knowledge base and mock booking.
- Deterministic rule engine designed for triage safety demonstration, not real medical diagnosis.

### Future Improvements
1. Expand validated medical knowledge base.
2. Connect to real EHR / FHIR scheduling systems.
3. Add HIPAA-compliant authentication and logging.